In [1]:
import pandas as pd
from transformers import pipeline
import torch

torch.cuda.is_available()

c:\Users\Kaylen\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1117 18:42:30.489000 37408 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


True

In [2]:
# Read in my data (parquet file)
posts_data = pd.read_parquet("data/clean_posts.parquet") # text is in 'full_text' column
posts_data.rename(columns={'full_text': 'document_text', 'link_flair_text': 'post_flair'}, inplace=True)
posts_data['document_id'] = posts_data['post_id']
posts_data['document_type'] = "post"
posts_data['link'] = "https://www.reddit.com/comments/" + posts_data["post_id"]

comments_data = pd.read_parquet("data/clean_comments.parquet") # text is in 'body' column
comments_data.rename(columns={'body': 'document_text'}, inplace=True)
comments_data['document_id'] = comments_data['comment_id']
comments_data['link'] = "https://www.reddit.com" + comments_data['permalink']
comments_data['document_type'] = "comment"

# Assigning comment records a post_flair value based on the post they belong to.
comments_data = comments_data.merge(
    posts_data[["post_id", "post_flair"]],  # right table: id and post_flair (additional column after join)
    on="post_id", how="left", validate="many_to_one"
)

common_columns = list(set(posts_data.columns).intersection(set(comments_data.columns)))
keep_columns_and_order = [
    "document_id",
    "document_type",
    "post_id",
    "parent_id",
    "created_utc",
    "post_flair",
    "author",
    "mains",
    "document_text",
    "score",
    "num_comments",
    "link",
    "subreddit",
]

# Append posts + comments into one table
full_data = pd.concat(
    [posts_data.reindex(columns=keep_columns_and_order),
    comments_data.reindex(columns=keep_columns_and_order)],
    ignore_index=True,
)

full_data["num_comments"] = full_data["num_comments"].astype("Int64")  # keep ints; NA for comments

# Post / top-level comment / reply
full_data["reply_level"] = "post"
full_data.loc[full_data["parent_id"].str.startswith("t3_", na=False), "reply_level"] = "top-level comment"
full_data.loc[full_data["parent_id"].str.startswith("t1_", na=False), "reply_level"] = "reply"


In [3]:
full_data["word_count"] = full_data["document_text"].str.split().str.len()

short_docs = full_data[full_data["word_count"] <= 5]
print(len(short_docs), "short docs")

# A random sample of 30 (re-run for a different sample)
short_docs[["document_type", "post_flair", "document_text"]].sample(30)

6507 short docs


,document_type,post_flair,document_text
18126,comment,Official Media,loveanddeepspace-en@infoldgames.com
15408,comment,Discussion,I hope so too :red_heart:
36147,comment,Discussion,thankssss
7953,comment,Discussion,Yes. The 6th
15767,comment,Discussion,Looks like they canceled Valko
49457,comment,Official Media,Just you bud.
40365,comment,Art,We were denied this…evil.
7700,comment,Art,heehee saving
14462,comment,Discussion,We were robbed
7970,comment,Discussion,Hear me out...


In [5]:
emotion_pipeline = pipeline("text-classification", model="SamLowe/roberta-base-go_emotions")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1791.92it/s]
